In [1]:
import sys
# Change to repository root, so it can find the config folder with the paths inside it
GITHUB_REPO_PATH = '/Users/user/Storage/Salk_Plant_Imaging/eckerlabproj'
# Add config folder with paths to Python's import search path
if GITHUB_REPO_PATH not in sys.path:
    sys.path.insert(0, GITHUB_REPO_PATH)

import config.paths as paths

In [2]:
# only needed in Jupyter Lab to see the images inline
%matplotlib widget

from plantcv import plantcv as pcv
from plantcv.parallel import WorkflowInputs
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import json
import shutil
import cv2

In [3]:
tray_number = "4"
stress = "Severe_Recovery"

# For grabbing the image: Finds an image in the image_holder
image_dir = os.path.join(paths.cm_image_directory, f'Tray{tray_number}_Holder')
files = [f for f in os.listdir(image_dir) if not f.startswith(".") if f.endswith(".jpg")]
if len(files) == 0:
    raise FileNotFoundError(f"No files found in {image_dir}.")
elif len(files) > 1:
    print(f"Warning: More than one image in {image_dir}, using one.")
filename = min(files, key=lambda f: os.path.getmtime(os.path.join(image_dir, f)))
image_name = filename
print(f"Image Name: {image_name}")
image_path = os.path.join(image_dir, image_name)
# Change to Chamber folder: this is where the temporary data on pixel number in plants is saved
t4_temp_chamber_image_results = paths.t4_temp_pixel_results
# Change to location where data from this run is saved
analysis_results_csv_path = paths.t4_analysis_results
# Change to location where plant_names.csv is saved
plant_names_path = paths.t4_plant_names
# Splitting filename to get the time
name_without_ext, _ = os.path.splitext(filename)
# print("\n" + name_without_ext)

name_parts = name_without_ext.split('_')
# print(name_parts)
time = name_parts[2]
print("\nImage Time: " + time)

FileNotFoundError: No files found in /Users/user/Library/CloudStorage/GoogleDrive-salkimager@gmail.com/My Drive/Chamber/CMiddle_Holder/Tray4_Holder.

In [ ]:
# Input/output options for PlantCV
args = WorkflowInputs(
    images=[image_path],
    names="image",
    result=t4_temp_chamber_image_results, 
    outdir=".",
    writeimg=False,
    debug="plot"
    )

# Set debug to the global parameter 
pcv.params.debug = args.debug
# Change display settings
pcv.params.dpi = 100
# Increase text size and thickness to make labels clearer
# (size may need to be altered based on original image size)
pcv.params.text_size = 10
pcv.params.text_thickness = 20

In [ ]:
# Shows your image, defines it as "img"
img, path, filename = pcv.readimage(filename=args.image)

In [ ]:
mtx_fname = os.path.join(paths.cm_fisheye_correction_matrices, 'mtx.npz')
dist_fname = os.path.join(paths.cm_fisheye_correction_matrices, 'dist.npz')

calibrated_img = pcv.transform.calibrate_camera(rgb_img=img, mtx_filename = mtx_fname, dist_filename = dist_fname)

In [ ]:
# Corrects image based on color standard (white of the power extension box) and stores output as corrected_img (roi=(x=, y=, w=, h=))
corrected_img = pcv.white_balance(calibrated_img, mode='hist', roi=(2740, 1977, 20, 20))

In [ ]:
# Rotates image
rotate_img = pcv.transform.rotate(corrected_img, -1, True)

In [ ]:
# Crops your image
crop_img = pcv.crop(img=rotate_img, x=1050, y=320, w=820, h=1630)

In [ ]:
# Masking based on LAB
# Shows options for which channel to view the image through (ideally want the most contrast)
colorspace_img = pcv.visualize.colorspaces(rgb_img=crop_img)

# Actually picks a channel with rgb2gray_"lab" and then the channel is the letter from "lab" that you look through
channeled_img = pcv.rgb2gray_lab(rgb_img=crop_img, channel='a')

# Masks the image
thresh_mask = pcv.threshold.binary(gray_img=channeled_img, threshold=129, object_type='dark')
cleaned_mask = pcv.fill(bin_img=thresh_mask, size=100)

# Fills in holes inside plant outline
filled_mask = pcv.fill_holes(bin_img=cleaned_mask)

In [ ]:
rois = pcv.roi.multi(img=filled_mask, 
                    coord=[(113,138), (324,140), (531,133), (702,161), 
                           (117,395), (306,405), (541,409), (697,432), 
                           (108,671), (360,681), (559,681), (700,674), 
                           (90,940), (364,934), (518,948), (738,948), 
                           (110,1200), (311,1209), (518,1214), (711,1242), 
                           (130,1500), (338,1500), (559,1481), (748,1504)],
                    radius=50)
labeled_mask, num_plants = pcv.create_labels(mask=filled_mask, rois=rois, roi_type='partial')

In [ ]:
# Outputs analyzed image
shape_image = pcv.analyze.size(img=crop_img, labeled_mask=labeled_mask, n_labels=num_plants)

In [ ]:
# Saves results for the 1 image (running again overwites past results)
pcv.outputs.save_results(filename= args.result, outformat="json")

In [ ]:
scale_values_path = paths.t4_scale_values_path
# Gets pixels to mm scalar s (mm per pixel, so: pixels * s^2 = mm^2)
with open(scale_values_path, "r") as f:
    scale_data = json.load(f)

mean_s = scale_data["mean_scale_mm2_per_pixel"]
std_s = scale_data["std_scale_mm2_per_pixel"]

In [ ]:
if os.path.isfile(analysis_results_csv_path):
    existing = pd.read_csv(analysis_results_csv_path, dtype={"Tray Number": str})
    already_recorded = (
        (existing["time"] == time) & 
        (existing["Tray Number"] == tray_number)
    ).any()
    if already_recorded:
        raise RuntimeError(
            f"Results for tray {tray_number} at time {time} already exist in CSV. "
            f"Skipping to avoid duplicates."
        )
        
# Create single-row DataFrame and write it to an analysis results csv
df = pd.read_csv(plant_names_path)
#Counts number of plants:
number_of_plants = df.shape[0]
for i in list(range(1,number_of_plants+1)): # starts at first number, goes up to but not including second number
    pixels_value = pcv.outputs.observations[f"default_{i}"]["area"]["value"]
    if pixels_value == 0:
        continue
    plant_number = f"plant{i}"
    plant_ID = df.loc[df["plant_number"] == plant_number, "plant_ID"].values[0]
    # stress = df.loc[df["plant_number"] == plant_number, "stress"].values[0]
    new_data = pd.DataFrame([{
        "time": time,
        "date": time[:10],
        "Tray Number": tray_number,
        "plant_number": plant_number,
        "plant_spec": f"tray{tray_number}_" + plant_number,
        "plant_ID": plant_ID,
        "Area (mm^2)": pixels_value * mean_s,
        "Stress (%)": stress,
    }])
    # If analysis_log.csv doesn't exist, creates it with a header:
    if not os.path.isfile(analysis_results_csv_path):
        new_data.to_csv(analysis_results_csv_path, index=False)
    else:
        # Append without writing header again
        new_data.to_csv(analysis_results_csv_path, mode='a', header=False, index=False)

In [ ]:
# Moves image to holder for after Tray4
short_term_dir = image_dir
long_term_dir = '/Users/user/Library/CloudStorage/GoogleDrive-salkimager@gmail.com/My Drive/Chamber/CMiddle_Holder/Tray5_Holder'
filename = image_name

# Full paths
src_path = os.path.join(short_term_dir, filename)
dst_path = os.path.join(long_term_dir, filename)

# Ensure long-term folder exists
os.makedirs(long_term_dir, exist_ok=True)

# Moves file to long-term folder
shutil.move(src_path, dst_path)

In [ ]:
plt.close('all')